# 🎓 Classroom Object Detection — YOLOv8 Training

> **Google Drive Dataset:** `ColabNotebooks/classroom_dataset`

## 📋 Notebook Flow
1. ✅ GPU Check
2. 📦 Install Dependencies
3. 📂 Mount Google Drive & Setup Paths
4. 🔍 Dataset Validation & Preview
5. 🚀 Train YOLOv8
6. 📊 View Results
7. ✅ Validate Model
8. 💾 Download Trained Model

---
## ⚙️ Step 1: GPU Check

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '❌ No GPU! Go to Runtime > Change runtime type > GPU')

import torch
print(f'\n🔥 PyTorch: {torch.__version__}')
print(f'🖥️  CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'🎮 GPU: {torch.cuda.get_device_name(0)}')
    print(f'💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

---
## 📦 Step 2: Install Dependencies

In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO
import ultralytics
print(f'✅ Ultralytics version: {ultralytics.__version__}')
ultralytics.checks()

---
## 📂 Step 3: Mount Google Drive & Setup Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('✅ Google Drive Mounted!')

In [ ]:
import os
from pathlib import Path

# ====================================================
# 🔧 CONFIGURE YOUR DATASET PATH HERE
# ====================================================
DRIVE_DATASET_PATH = '/content/drive/MyDrive/ColabNotebooks/classroom_dataset'
# ====================================================

dataset_path = Path(DRIVE_DATASET_PATH)

if not dataset_path.exists():
    print(f'❌ Dataset not found at: {dataset_path}')
    print('📌 Update DRIVE_DATASET_PATH above!')
else:
    print(f'✅ Dataset found at: {dataset_path}')
    print('\n📁 Dataset Structure:')
    for item in sorted(dataset_path.rglob('*')):
        if item.is_dir():
            count = len(list(item.glob('*')))
            rel = item.relative_to(dataset_path)
            print(f'   📂 {rel}/ ({count} items)')
        elif item.suffix in ['.txt', '.yaml', '.json'] and item.parent == dataset_path:
            print(f'   📄 {item.name}')

In [ ]:
import yaml, json

# Read classes.txt
with open(dataset_path / 'classes.txt', 'r') as f:
    classes = [line.strip() for line in f if line.strip()]

print(f'📋 Classes ({len(classes)} total):')
for i, cls in enumerate(classes):
    print(f'   [{i}] {cls}')

# Read data.yaml
with open(dataset_path / 'data.yaml', 'r') as f:
    data_config = yaml.safe_load(f)

print(f'\n📄 data.yaml content:')
for key, val in data_config.items():
    print(f'   {key}: {val}')

# Read notes.json
notes_file = dataset_path / 'notes.json'
if notes_file.exists():
    with open(notes_file) as f:
        notes = json.load(f)
    print(f'\n📝 Dataset Notes:')
    print(json.dumps(notes, indent=2))

---
## 🔧 Step 4: Fix data.yaml & Validate Dataset

In [ ]:
import yaml
from pathlib import Path

fixed_yaml_path = '/content/classroom_data.yaml'
train_images = str(dataset_path / 'images' / 'train')
val_images   = str(dataset_path / 'images' / 'val')

train_count = len(list(Path(train_images).glob('*.jpg'))) + len(list(Path(train_images).glob('*.png')))
val_count   = len(list(Path(val_images).glob('*.jpg')))   + len(list(Path(val_images).glob('*.png')))

fixed_config = {
    'path'  : str(dataset_path),
    'train' : train_images,
    'val'   : val_images,
    'nc'    : len(classes),
    'names' : classes
}

with open(fixed_yaml_path, 'w') as f:
    yaml.dump(fixed_config, f, default_flow_style=False)

print('✅ Fixed data.yaml created!')
print(f'\n📊 Dataset Summary:')
print(f'   🏋️  Train images : {train_count}')
print(f'   🧪 Val images   : {val_count}')
print(f'   🏷️  Classes      : {len(classes)}')

with open(fixed_yaml_path) as f:
    print('\n📄 Fixed data.yaml:')
    print(f.read())

In [ ]:
import random, cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

train_img_dir = Path(train_images)
labels_dir    = dataset_path / 'labels' / 'train'
img_files     = list(train_img_dir.glob('*.jpg')) + list(train_img_dir.glob('*.png'))
sample_imgs   = random.sample(img_files, min(6, len(img_files)))
colors        = plt.cm.Set1(np.linspace(0, 1, max(len(classes), 1)))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('📸 Sample Training Images with Annotations', fontsize=16, fontweight='bold')
axes = axes.flatten()

for idx, img_path in enumerate(sample_imgs):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    ax = axes[idx]
    ax.imshow(img)

    label_path = labels_dir / (img_path.stem + '.txt')
    if label_path.exists():
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls_id, cx, cy, bw, bh = int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                    x1 = (cx - bw/2) * w
                    y1 = (cy - bh/2) * h
                    color = colors[cls_id % len(colors)]
                    rect = patches.Rectangle((x1, y1), bw*w, bh*h, linewidth=2, edgecolor=color, facecolor='none')
                    ax.add_patch(rect)
                    cls_name = classes[cls_id] if cls_id < len(classes) else str(cls_id)
                    ax.text(x1, y1-4, cls_name, color='white', fontsize=8,
                            bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.8))

    ax.set_title(img_path.name[:30], fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('/content/sample_annotations.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Sample annotations saved!')

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

def count_labels(label_dir):
    counter = Counter()
    for lf in Path(label_dir).glob('*.txt'):
        with open(lf) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    counter[int(parts[0])] += 1
    return counter

train_dist = count_labels(dataset_path / 'labels' / 'train')
val_dist   = count_labels(dataset_path / 'labels' / 'val')

print('📊 Class Distribution (Train):')
for cls_id in sorted(train_dist.keys()):
    cls_name = classes[cls_id] if cls_id < len(classes) else str(cls_id)
    bar = '█' * (train_dist[cls_id] // 10 + 1)
    print(f'   [{cls_id}] {cls_name:<20} {train_dist[cls_id]:>5}  {bar}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, dist, title in [(axes[0], train_dist, 'Train'), (axes[1], val_dist, 'Validation')]:
    ids   = sorted(dist.keys())
    names = [classes[i] if i < len(classes) else str(i) for i in ids]
    vals  = [dist[i] for i in ids]
    bars  = ax.bar(names, vals, color=plt.cm.viridis(np.linspace(0.2, 0.9, len(ids))))
    ax.set_title(f'{title} Set — Class Distribution', fontweight='bold')
    ax.set_xlabel('Class')
    ax.set_ylabel('Instances')
    ax.tick_params(axis='x', rotation=45)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(val), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('/content/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🚀 Step 5: Train YOLOv8

| Parameter | Options | Default |
|-----------|---------|--------|
| `MODEL_SIZE` | `n` `s` `m` `l` `x` | `s` |
| `EPOCHS` | 50–300 | `100` |
| `IMG_SIZE` | 416, 640, 1280 | `640` |
| `BATCH` | -1=auto, 8, 16, 32 | `-1` |

In [ ]:
# ====================================================
# 🔧 TRAINING CONFIGURATION
# ====================================================
MODEL_SIZE = 's'      # n=nano  s=small  m=medium  l=large  x=xlarge
EPOCHS     = 100      # Training epochs
IMG_SIZE   = 640      # Input image size
BATCH      = -1       # -1 = auto (recommended)
PROJECT    = '/content/drive/MyDrive/ColabNotebooks/classroom_dataset/runs'
RUN_NAME   = 'classroom_yolov8'
PATIENCE   = 20       # Early stopping patience
WORKERS    = 2        # Dataloader workers
DEVICE     = 0        # 0=GPU  'cpu'=CPU
# ====================================================

print('🔧 Training Configuration:')
print(f'   Model     : YOLOv8{MODEL_SIZE}')
print(f'   Epochs    : {EPOCHS}')
print(f'   Image Size: {IMG_SIZE}x{IMG_SIZE}')
print(f'   Batch     : {"Auto" if BATCH == -1 else BATCH}')
print(f'   Device    : {"GPU" if DEVICE == 0 else "CPU"}')
print(f'   Run dir   : {PROJECT}/{RUN_NAME}')

In [ ]:
from ultralytics import YOLO
import time

model = YOLO(f'yolov8{MODEL_SIZE}.pt')
print(f'✅ Loaded YOLOv8{MODEL_SIZE} — {sum(p.numel() for p in model.model.parameters()):,} params')

print('\n🚀 Starting Training...')
print('=' * 60)
start_time = time.time()

results = model.train(
    data         = fixed_yaml_path,
    epochs       = EPOCHS,
    imgsz        = IMG_SIZE,
    batch        = BATCH,
    device       = DEVICE,
    project      = PROJECT,
    name         = RUN_NAME,
    patience     = PATIENCE,
    workers      = WORKERS,
    pretrained   = True,
    optimizer    = 'AdamW',
    lr0          = 0.001,
    lrf          = 0.01,
    momentum     = 0.937,
    weight_decay = 0.0005,
    warmup_epochs= 3,
    augment      = True,
    mosaic       = 1.0,
    mixup        = 0.1,
    copy_paste   = 0.1,
    degrees      = 5.0,
    translate    = 0.1,
    scale        = 0.5,
    fliplr       = 0.5,
    flipud       = 0.0,
    save         = True,
    save_period  = 10,
    plots        = True,
    exist_ok     = True,
    verbose      = True,
    seed         = 42,
)

elapsed = time.time() - start_time
print('\n' + '=' * 60)
print(f'✅ Training Complete! Time: {elapsed/3600:.1f}h ({elapsed/60:.0f} min)')

run_dir = Path(PROJECT) / RUN_NAME
print(f'📁 Results: {run_dir}')

---
## 📊 Step 6: View Training Results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

run_dir = Path(PROJECT) / RUN_NAME

# Training curves
results_png = run_dir / 'results.png'
if results_png.exists():
    plt.figure(figsize=(20, 8))
    img = mpimg.imread(str(results_png))
    plt.imshow(img)
    plt.axis('off')
    plt.title('📈 Training Results', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_files = [
    ('confusion_matrix_normalized.png', 'Normalized Confusion Matrix'),
    ('PR_curve.png',                    'Precision-Recall Curve'),
    ('F1_curve.png',                    'F1 Score Curve'),
    ('P_curve.png',                     'Precision Curve'),
    ('R_curve.png',                     'Recall Curve'),
    ('confusion_matrix.png',            'Confusion Matrix'),
]

available = [(name, title) for name, title in plot_files if (run_dir / name).exists()]

if available:
    cols = 3
    rows = (len(available) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(18, 5 * rows))
    axes = np.array(axes).flatten()

    for i, (name, title) in enumerate(available):
        img = mpimg.imread(str(run_dir / name))
        axes[i].imshow(img)
        axes[i].set_title(title, fontweight='bold')
        axes[i].axis('off')

    for j in range(len(available), len(axes)):
        axes[j].axis('off')

    plt.suptitle('📊 Evaluation Plots', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd

csv_path = run_dir / 'results.csv'
if csv_path.exists():
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    if 'metrics/mAP50(B)' in df.columns:
        best_epoch = df['metrics/mAP50(B)'].idxmax()
        best_row   = df.iloc[best_epoch]
        print(f'🏆 Best Epoch: {int(best_row.get("epoch", best_epoch))}')

        metrics = {
            'mAP50'    : 'metrics/mAP50(B)',
            'mAP50-95' : 'metrics/mAP50-95(B)',
            'Precision': 'metrics/precision(B)',
            'Recall'   : 'metrics/recall(B)',
        }
        for name, col in metrics.items():
            if col in df.columns:
                val = best_row[col]
                bar = '█' * int(val * 30) + '░' * (30 - int(val * 30))
                print(f'   {name:<12}: {val:.4f}  [{bar}]')

    print(f'\n📋 Last 5 Epochs:')
    display_cols = [c for c in ['epoch','metrics/mAP50(B)','metrics/mAP50-95(B)',
                                'metrics/precision(B)','metrics/recall(B)'] if c in df.columns]
    print(df[display_cols].tail(5).to_string(index=False))

---
## ✅ Step 7: Validate & Test Model

In [ ]:
from ultralytics import YOLO

best_model_path = run_dir / 'weights' / 'best.pt'
print(f'📦 Loading: {best_model_path}')
best_model = YOLO(str(best_model_path))

val_results = best_model.val(
    data    = fixed_yaml_path,
    imgsz   = IMG_SIZE,
    device  = DEVICE,
    verbose = True,
)

print('\n✅ Validation Results:')
print(f'   mAP50     : {val_results.box.map50:.4f}')
print(f'   mAP50-95  : {val_results.box.map:.4f}')
print(f'   Precision : {val_results.box.mp:.4f}')
print(f'   Recall    : {val_results.box.mr:.4f}')

In [ ]:
import random, cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

val_img_dir = Path(val_images)
val_imgs    = list(val_img_dir.glob('*.jpg')) + list(val_img_dir.glob('*.png'))
sample_val  = random.sample(val_imgs, min(4, len(val_imgs)))

results_list = best_model.predict(
    source  = [str(p) for p in sample_val],
    conf    = 0.25,
    iou     = 0.45,
    imgsz   = IMG_SIZE,
    device  = DEVICE,
    save    = False,
    verbose = False,
)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('🔍 Model Predictions on Validation Images', fontsize=16, fontweight='bold')
axes = axes.flatten()
cmap = plt.cm.Set1(np.linspace(0, 1, max(len(classes), 1)))

for idx, (res, ax) in enumerate(zip(results_list, axes)):
    img = cv2.imread(str(sample_val[idx]))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)

    if res.boxes is not None and len(res.boxes) > 0:
        for box in res.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            cls_id   = int(box.cls[0].cpu().numpy())
            conf     = float(box.conf[0].cpu().numpy())
            color    = cmap[cls_id % len(cmap)]
            cls_name = classes[cls_id] if cls_id < len(classes) else str(cls_id)
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor=color, facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1-4, f'{cls_name} {conf:.2f}', color='white', fontsize=9,
                    bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.85))
        ax.set_title(f'{sample_val[idx].name[:25]} — {len(res.boxes)} detections', fontsize=10)
    else:
        ax.set_title(f'{sample_val[idx].name[:25]} — No detections', fontsize=10)

    ax.axis('off')

plt.tight_layout()
plt.savefig('/content/val_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved to /content/val_predictions.png')

---
## 💾 Step 8: Export & Download Model

In [ ]:
from pathlib import Path

best_pt = run_dir / 'weights' / 'best.pt'
last_pt = run_dir / 'weights' / 'last.pt'

print('📦 Model Files (saved in Google Drive):')
for f in [best_pt, last_pt]:
    if f.exists():
        size_mb = f.stat().st_size / 1e6
        print(f'   ✅ {f.name}: {size_mb:.1f} MB')
        print(f'      Path: {f}')

print(f'\n📌 Next steps:')
print(f'   1. Download best.pt from Google Drive → runs/{RUN_NAME}/weights/best.pt')
print(f'   2. Place in backend/models/best.pt')
print(f'   3. Run: uvicorn main:app --reload')

In [ ]:
# Download best.pt directly to your computer
from google.colab import files
import shutil

local_best = '/content/classroom_best.pt'
shutil.copy(str(best_pt), local_best)

print('⬇️  Downloading classroom_best.pt to your computer...')
files.download(local_best)
print('✅ Download started!')

In [ ]:
# Optional: Export to ONNX (faster inference on CPU)
print('📤 Exporting to ONNX...')
onnx_path = best_model.export(format='onnx', imgsz=IMG_SIZE, simplify=True)
print(f'✅ ONNX exported: {onnx_path}')

from google.colab import files
import shutil
onnx_dest = '/content/classroom_best.onnx'
shutil.copy(str(onnx_path), onnx_dest)
files.download(onnx_dest)
print('✅ ONNX download started!')

---
## 📋 Final Summary

In [ ]:
import pandas as pd

print('=' * 60)
print('🎓 CLASSROOM YOLO TRAINING — FINAL SUMMARY')
print('=' * 60)
print(f'\n📂 Dataset      : {dataset_path}')
print(f'🏋️  Train images : {train_count}')
print(f'🧪 Val images   : {val_count}')
print(f'🏷️  Classes ({len(classes)}): {classes}')
print(f'\n🤖 Model        : YOLOv8{MODEL_SIZE}')
print(f'📏 Image Size   : {IMG_SIZE}x{IMG_SIZE}')
print(f'🔄 Epochs       : {EPOCHS}')

csv_path = run_dir / 'results.csv'
if csv_path.exists():
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    if 'metrics/mAP50(B)' in df.columns:
        best_epoch = df['metrics/mAP50(B)'].idxmax()
        best_row   = df.iloc[best_epoch]
        print(f'\n🏆 Best Epoch   : {int(best_row.get("epoch", best_epoch))}')
        print(f'   mAP50        : {best_row["metrics/mAP50(B)"]:.4f}')
        if 'metrics/mAP50-95(B)' in df.columns:
            print(f'   mAP50-95     : {best_row["metrics/mAP50-95(B)"]:.4f}')

print(f'\n💾 Model Path   :')
print(f'   {run_dir}/weights/best.pt')
print(f'\n✅ Done! Place best.pt in backend/models/ to use.')
print('=' * 60)